# 13 · Opciones de diseño y colorimetría del dashboard

Seis propuestas para el tablero, cada una en **modo claro** y **modo oscuro**. Cada vista previa usa los datos
reales del pronóstico de 24 h (ensamble + bandas conformes) y las señales del motor para el **generador**,
con los mismos componentes del dashboard: tarjeta principal, indicadores con minigráfica, franja de 24 horas,
mapa de calor día-hora y pronóstico con banda.

Las opciones cambian más que el color. También varían la **tipografía**, el **radio de las esquinas**, las **sombras** y el
**estilo de la tarjeta principal** (borde lateral, fondo teñido o barra superior).

Para elegir, basta con decir, por ejemplo, *"opción D"* o *"la C en claro con la F en oscuro"*. Las opciones pasan
como tokens al diccionario `TEMAS` de `dashboard/app.py`, así que se pueden mezclar.

Al final hay dos chequeos objetivos:
- el **contraste WCAG** del texto y de las señales sobre el panel;
- una **simulación de daltonismo** (deuteranopía y protanopía), porque verde frente a rojo es justo la distinción que más falla.

> Las vistas previas son HTML. Se ven completas en VS Code o en Jupyter; GitHub puede quitar parte de los estilos.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import HTML, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from motor_decision import (cargar_fuente_pronostico, comparar_metodos_estable,
                            elegir_mejor_metodo_estable, generar_senales)

bandas, meta = cargar_fuente_pronostico(RAIZ, "24h")
hist = pd.read_csv(RAIZ / "data/processed/dataset_maestro_2019_2025.csv", usecols=["precio_bolsa"])["precio_bolsa"].values
ROL = "generador"
metodo, _ = elegir_mejor_metodo_estable(comparar_metodos_estable(bandas, ROL, hist))
df = bandas.assign(senal=generar_senales(bandas, metodo, ROL, hist)).copy()
df["dia"], df["hora"] = df["fecha_hora"].dt.date, df["fecha_hora"].dt.hour
completos = [d for d, g in df.groupby("dia") if len(g) == 24]
DIA = completos[-1]
HORA = 19
hoy = df[df["dia"] == DIA].set_index("hora")
fila = hoy.loc[HORA]

diario = df[df["dia"].isin(completos)].groupby("dia").agg(
    q50=("q50", "mean"), ancho=("q90", "mean"), q10=("q10", "mean"))
diario["ancho"] = diario["ancho"] - diario["q10"]
cubre = ((df["real"] >= df["q10"]) & (df["real"] <= df["q90"])).groupby(df["dia"]).mean() * 100
diario["cobertura"] = cubre.reindex(diario.index).rolling(7, min_periods=1).mean()
ult = diario.tail(14)

DATOS = {
    "dia": DIA, "hora": HORA, "senal": fila["senal"], "q50": fila["q50"], "q10": fila["q10"], "q90": fila["q90"],
    "franja": [hoy.loc[h, "senal"] for h in range(24)],
    "calor": [[df[(df["dia"] == d) & (df["hora"] == h)]["senal"].iloc[0] for h in range(24)] for d in completos[-14:]],
    "dias_calor": completos[-14:],
    "spark": {"Precio medio esperado": (ult["q50"].tolist(), f"{ult['q50'].iloc[-1]:.0f} COP/kWh"),
              "Ancho de banda medio": (ult["ancho"].tolist(), f"{ult['ancho'].iloc[-1]:.0f} COP/kWh"),
              "Cobertura móvil 7 d": (ult["cobertura"].tolist(), f"{ult['cobertura'].iloc[-1]:.0f} %")},
    "serie": df[df["dia"].isin(completos[-3:])][["fecha_hora", "real", "q50", "q10", "q90"]].reset_index(drop=True),
}
print(f"Método del generador: {metodo} · día {DIA:%d/%m/%Y} {HORA}:00 → señal '{fila['senal']}'")

[validar_contrato_pronostico] advertencia: C:\Users\mgdbj\xm-spot-price-predictor\data\processed\resultados\pronostico_ensamble_bandas_24h_2026.csv tiene 87 fila(s) con cuantiles cruzados (q10>q50 o q50>q90) de 4849 totales -- revisar si crece en un modelo nuevo, probablemente borde de calibración.


Método del generador: hibrido · día 04/08/2026 19:00 → señal 'vender'


## Las seis opciones

| | Nombre | Idea | Tipografía | Tarjeta principal |
|---|---|---|---|---|
| A | **Pizarra (actual)** | La paleta vigente: grises azulados de Tailwind, azul como acento | Sistema (Source Sans) | Borde lateral |
| B | **Energía** | Azul marino institucional con ámbar de acento; "esperar" pasa a gris para no chocar con el ámbar | IBM Plex Sans | Barra superior |
| C | **Grafito** | Grises neutros (zinc), acento turquesa, sin sombras y esquinas medianas | Inter | Fondo teñido |
| D | **Accesible (Okabe-Ito)** | Paleta segura para daltonismo: azul = actuar, amarillo/naranja = esperar, bermellón = evitar | Source Sans 3 | Borde lateral |
| E | **Nórdico** | Baja saturación (paleta Nord), mucho aire, bordes finos y sin sombras | Nunito Sans | Fondo teñido |
| F | **Sala de control** | Estética SCADA o de instrumentación: casi negro, señales intensas, cifras en monoespaciada | IBM Plex Sans + JetBrains Mono | Barra superior |

In [2]:
OPCIONES = {
 "A": dict(nombre="Pizarra (actual)", fuente="'Source Sans 3', 'Segoe UI', sans-serif", fuente_num=None,
           radio=14, sombra=True, hero="borde",
           claro=dict(fondo="#F1F5F9", panel="#FFFFFF", linea="#E2E8F0", tinta="#0F172A", texto="#334155", gris="#64748B",
                      acento="#2563EB", banda="rgba(37,99,235,0.13)", verde="#15803D", ambar="#B45309", rojo="#B91C1C",
                      verde_bg="#DCFCE7", ambar_bg="#FEF3C7", rojo_bg="#FEE2E2",
                      calor_verde="#16A34A", calor_ambar="#F59E0B", calor_rojo="#DC2626"),
           oscuro=dict(fondo="#0A1120", panel="#111A2E", linea="#24304A", tinta="#E6EAF2", texto="#C3CBD9", gris="#8D99AE",
                       acento="#60A5FA", banda="rgba(96,165,250,0.20)", verde="#34D399", ambar="#FBBF24", rojo="#F87171",
                       verde_bg="rgba(52,211,153,0.16)", ambar_bg="rgba(251,191,36,0.16)", rojo_bg="rgba(248,113,113,0.16)",
                       calor_verde="#34D399", calor_ambar="#FBBF24", calor_rojo="#F87171")),
 "B": dict(nombre="Energía", fuente="'IBM Plex Sans', 'Segoe UI', sans-serif", fuente_num=None,
           radio=10, sombra=True, hero="barra",
           claro=dict(fondo="#F4F6FA", panel="#FFFFFF", linea="#DDE3EC", tinta="#0B1F3A", texto="#2B3D57", gris="#5B6B82",
                      acento="#D97706", banda="rgba(30,58,138,0.12)", verde="#047857", ambar="#475569", rojo="#C2410C",
                      verde_bg="#D1FAE5", ambar_bg="#E2E8F0", rojo_bg="#FFEDD5",
                      calor_verde="#059669", calor_ambar="#CBD5E1", calor_rojo="#EA580C", linea_q50="#1E3A8A"),
           oscuro=dict(fondo="#071426", panel="#0D1F38", linea="#1D3355", tinta="#EAF0F8", texto="#BCC8DA", gris="#8193AD",
                       acento="#FBBF24", banda="rgba(251,191,36,0.16)", verde="#34D399", ambar="#94A3B8", rojo="#FB923C",
                       verde_bg="rgba(52,211,153,0.15)", ambar_bg="rgba(148,163,184,0.16)", rojo_bg="rgba(251,146,60,0.16)",
                       calor_verde="#10B981", calor_ambar="#334155", calor_rojo="#F97316", linea_q50="#FBBF24")),
 "C": dict(nombre="Grafito", fuente="'Inter', 'Segoe UI', sans-serif", fuente_num=None,
           radio=10, sombra=False, hero="tenido",
           claro=dict(fondo="#FAFAFA", panel="#FFFFFF", linea="#E4E4E7", tinta="#18181B", texto="#3F3F46", gris="#71717A",
                      acento="#0D9488", banda="rgba(13,148,136,0.13)", verde="#15803D", ambar="#A16207", rojo="#BE123C",
                      verde_bg="#F0FDF4", ambar_bg="#FEFCE8", rojo_bg="#FFF1F2",
                      calor_verde="#22C55E", calor_ambar="#EAB308", calor_rojo="#E11D48"),
           oscuro=dict(fondo="#09090B", panel="#18181B", linea="#27272A", tinta="#FAFAFA", texto="#D4D4D8", gris="#A1A1AA",
                       acento="#2DD4BF", banda="rgba(45,212,191,0.16)", verde="#4ADE80", ambar="#FACC15", rojo="#FB7185",
                       verde_bg="rgba(74,222,128,0.10)", ambar_bg="rgba(250,204,21,0.10)", rojo_bg="rgba(251,113,133,0.10)",
                       calor_verde="#22C55E", calor_ambar="#CA8A04", calor_rojo="#E11D48")),
 "D": dict(nombre="Accesible (Okabe-Ito)", fuente="'Source Sans 3', 'Segoe UI', sans-serif", fuente_num=None,
           radio=12, sombra=True, hero="borde",
           claro=dict(fondo="#F5F6F8", panel="#FFFFFF", linea="#E1E4EA", tinta="#111827", texto="#374151", gris="#6B7280",
                      acento="#0072B2", banda="rgba(0,114,178,0.13)", verde="#0072B2", ambar="#8A6100", rojo="#B54400",
                      verde_bg="#DCEEFA", ambar_bg="#FDF3D7", rojo_bg="#FBE3D6",
                      calor_verde="#0072B2", calor_ambar="#F0E442", calor_rojo="#D55E00"),
           oscuro=dict(fondo="#0E1117", panel="#161B24", linea="#2A3140", tinta="#EEF1F6", texto="#C6CCD8", gris="#8E97A8",
                       acento="#56B4E9", banda="rgba(86,180,233,0.20)", verde="#56B4E9", ambar="#F0E442", rojo="#FF8C42",
                       verde_bg="rgba(86,180,233,0.16)", ambar_bg="rgba(240,228,66,0.14)", rojo_bg="rgba(255,140,66,0.16)",
                       calor_verde="#3A9AD9", calor_ambar="#E8DC4A", calor_rojo="#E6731F")),
 "E": dict(nombre="Nórdico", fuente="'Nunito Sans', 'Segoe UI', sans-serif", fuente_num=None,
           radio=8, sombra=False, hero="tenido",
           claro=dict(fondo="#F7F7F4", panel="#FFFFFF", linea="#E5E7EB", tinta="#2E3440", texto="#434C5E", gris="#6B7385",
                      acento="#5E81AC", banda="rgba(94,129,172,0.14)", verde="#4F7A52", ambar="#9A6B12", rojo="#A3444E",
                      verde_bg="#EAF2E8", ambar_bg="#F8F0DC", rojo_bg="#F6E4E6",
                      calor_verde="#8DB38F", calor_ambar="#E5C07B", calor_rojo="#C97B84"),
           oscuro=dict(fondo="#2E3440", panel="#3B4252", linea="#4C566A", tinta="#ECEFF4", texto="#D8DEE9", gris="#AEB6C6",
                       acento="#88C0D0", banda="rgba(136,192,208,0.20)", verde="#A3BE8C", ambar="#EBCB8B", rojo="#E0838C",
                       verde_bg="rgba(163,190,140,0.18)", ambar_bg="rgba(235,203,139,0.16)", rojo_bg="rgba(224,131,140,0.18)",
                       calor_verde="#A3BE8C", calor_ambar="#EBCB8B", calor_rojo="#BF616A")),
 "F": dict(nombre="Sala de control", fuente="'IBM Plex Sans', 'Segoe UI', sans-serif",
           fuente_num="'JetBrains Mono', Consolas, monospace", radio=4, sombra=False, hero="barra",
           claro=dict(fondo="#ECEEEF", panel="#FAFAFA", linea="#C9CED3", tinta="#101418", texto="#2C3338", gris="#5D666E",
                      acento="#0E7490", banda="rgba(14,116,144,0.14)", verde="#15803D", ambar="#B45309", rojo="#B91C1C",
                      verde_bg="#DDF3E4", ambar_bg="#FBEBD3", rojo_bg="#F9DEDE",
                      calor_verde="#16A34A", calor_ambar="#F59E0B", calor_rojo="#DC2626"),
           oscuro=dict(fondo="#050607", panel="#0E1113", linea="#22282C", tinta="#E8ECEE", texto="#B8C1C6", gris="#7C878D",
                       acento="#22D3EE", banda="rgba(34,211,238,0.14)", verde="#22C55E", ambar="#F59E0B", rojo="#EF4444",
                       verde_bg="rgba(34,197,94,0.12)", ambar_bg="rgba(245,158,11,0.12)", rojo_bg="rgba(239,68,68,0.12)",
                       calor_verde="#16A34A", calor_ambar="#D97706", calor_rojo="#DC2626")),
}

In [3]:
SIGNIFICADO = {"comprar": ("COMPRAR EN BOLSA", "verde"), "vender": ("DESPACHAR Y VENDER", "verde"),
               "evitar_compra": ("EVITAR LA BOLSA", "rojo"), "retener": ("RETENER GENERACIÓN", "rojo"),
               "esperar": ("ESPERAR", "ambar")}
FUENTES = ("https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;600;700&family=Inter:wght@400;600;800"
           "&family=JetBrains+Mono:wght@500;700&family=Nunito+Sans:wght@400;700;800&family=Source+Sans+3:wght@400;600;800&display=swap")


def sparkline(vals, color, w=110, h=34):
    v = np.asarray(vals, float); lo, hi = v.min(), v.max(); rg = (hi - lo) or 1
    pts = " ".join(f"{i * w / (len(v) - 1):.1f},{h - 3 - (x - lo) / rg * (h - 6):.1f}" for i, x in enumerate(v))
    return (f'<svg width="{w}" height="{h}"><polyline points="{pts}" fill="none" stroke="{color}" stroke-width="2" '
            f'stroke-linejoin="round" stroke-linecap="round"/><circle cx="{w:.1f}" cy="{pts.split()[-1].split(",")[1]}" '
            f'r="3" fill="{color}"/></svg>')


def grafico_banda(serie, t, w=560, h=190):
    s = serie; n = len(s); lo = min(s["q10"].min(), s["real"].min()); hi = max(s["q90"].max(), s["real"].max())
    X = lambda i: 34 + i * (w - 44) / (n - 1)
    Y = lambda y: 10 + (hi - y) / (hi - lo) * (h - 34)
    banda = " ".join(f"{X(i):.1f},{Y(y):.1f}" for i, y in enumerate(s["q90"])) + " " + \
            " ".join(f"{X(i):.1f},{Y(y):.1f}" for i, y in reversed(list(enumerate(s["q10"]))))
    linea = lambda col: " ".join(f"{X(i):.1f},{Y(y):.1f}" for i, y in enumerate(s[col]))
    marcas = "".join(f'<line x1="{X(i)}" x2="{X(i)}" y1="10" y2="{h - 24}" stroke="{t["linea"]}"/>'
                     f'<text x="{X(i) + 3}" y="{h - 8}" font-size="10" fill="{t["gris"]}">{s["fecha_hora"][i]:%d/%m}</text>'
                     for i in range(0, n, 24))
    ejes = "".join(f'<text x="2" y="{Y(v) + 3:.1f}" font-size="10" fill="{t["gris"]}">{v:.0f}</text>'
                   for v in np.linspace(lo, hi, 4))
    q50 = t.get("linea_q50", t["acento"])
    return (f'<svg width="100%" viewBox="0 0 {w} {h}">{marcas}{ejes}<polygon points="{banda}" fill="{t["banda"]}"/>'
            f'<polyline points="{linea("q50")}" fill="none" stroke="{q50}" stroke-width="2"/>'
            f'<polyline points="{linea("real")}" fill="none" stroke="{t["tinta"]}" stroke-width="1.3" stroke-dasharray="3 2"/></svg>')


def celda(senal, t, sufijo=""):
    return t["calor_" + SIGNIFICADO[senal][1]] if sufijo == "calor" else t[SIGNIFICADO[senal][1]]


def vista_previa(op, modo, d=DATOS):
    o = OPCIONES[op]; t = o[modo]; r = o["radio"]; num = o["fuente_num"] or o["fuente"]
    sombra = ("0 1px 3px rgba(0,0,0,.35)" if modo == "oscuro" else "0 1px 2px rgba(15,23,42,.07)") if o["sombra"] else "none"
    caja = f"background:{t['panel']};border:1px solid {t['linea']};border-radius:{r}px;box-shadow:{sombra};"
    accion, tono = SIGNIFICADO[d["senal"]]; c = t[tono]; cbg = t[tono + "_bg"]
    if o["hero"] == "borde":
        hero_estilo = caja + f"border-left:7px solid {c};"
    elif o["hero"] == "tenido":
        hero_estilo = f"background:{cbg};border:1px solid {t['linea']};border-radius:{r}px;"
    else:
        hero_estilo = caja + f"border-top:5px solid {c};"
    pos = np.clip((d["q50"] - d["q10"]) / (d["q90"] - d["q10"]) * 100, 2, 98)
    hero = f"""
    <div style="{hero_estilo}padding:16px 18px;flex:1.25;min-width:250px">
      <div style="font-size:.7rem;letter-spacing:.06em;text-transform:uppercase;color:{t['gris']}">
        Recomendación · {d['dia']:%d/%m/%Y} {d['hora']:02d}:00 · generador</div>
      <div style="font-size:1.75rem;font-weight:800;color:{c};margin:.2rem 0 .35rem">{accion}</div>
      <div style="color:{t['texto']};font-size:.9rem">Precio esperado
        <b style="color:{t['tinta']};font-family:{num}">{d['q50']:.0f} COP/kWh</b>,
        banda del 80 % entre {d['q10']:.0f} y {d['q90']:.0f}.</div>
      <div style="position:relative;height:12px;border-radius:{min(r, 8)}px;margin:18px 0 6px;border:1px solid {t['linea']};
        background:linear-gradient(90deg,{t['calor_verde']}55,{t['calor_ambar']}55,{t['calor_rojo']}55)">
        <div style="position:absolute;left:{pos:.0f}%;top:-5px;width:3px;height:20px;background:{t['tinta']};border-radius:2px"></div></div>
      <div style="display:flex;justify-content:space-between;font-size:.7rem;color:{t['gris']};font-family:{num}">
        <span>q10 {d['q10']:.0f}</span><span>q90 {d['q90']:.0f}</span></div>
      <span style="display:inline-block;margin-top:10px;padding:2px 10px;border-radius:999px;font-size:.72rem;font-weight:700;
        background:{t['verde_bg']};color:{t['verde']}">Confianza alta</span>
    </div>"""
    tarjetas = "".join(f"""
      <div style="{caja}padding:10px 12px;flex:1;min-width:150px">
        <div style="font-size:.72rem;color:{t['gris']}">{nombre}</div>
        <div style="display:flex;justify-content:space-between;align-items:flex-end;gap:6px">
          <div style="font-size:1.2rem;font-weight:750;color:{t['tinta']};font-family:{num};white-space:nowrap">{valor}</div>
          {sparkline(serie, t['acento'])}</div></div>""" for nombre, (serie, valor) in d["spark"].items())
    franja = "".join(
        f'<div title="{h}:00" style="flex:1;height:30px;background:{celda(s, t, "calor")};border-radius:{min(r, 5)}px;'
        f'{"outline:2px solid " + t["tinta"] + ";outline-offset:1px;" if h == d["hora"] else ""}'
        f'display:flex;align-items:center;justify-content:center;font-size:.6rem;color:#fff;font-family:{num}">{h}</div>'
        for h, s in enumerate(d["franja"]))
    filas = "".join(
        f'<div style="display:flex;gap:2px;align-items:center"><span style="width:38px;font-size:.6rem;color:{t["gris"]};font-family:{num}">{dia:%d/%m}</span>'
        + "".join(f'<div style="flex:1;height:11px;border-radius:2px;background:{celda(s, t, "calor")}"></div>' for s in fila)
        + "</div>" for dia, fila in zip(d["dias_calor"], d["calor"]))
    ley = " ".join(f'<span style="display:inline-flex;align-items:center;gap:4px;margin-right:10px">'
                   f'<i style="width:10px;height:10px;border-radius:2px;background:{t["calor_" + k]};display:inline-block"></i>{v}</span>'
                   for k, v in (("verde", "Despachar y vender"), ("ambar", "Esperar"), ("rojo", "Retener")))
    return f"""
    <link href="{FUENTES}" rel="stylesheet">
    <div style="background:{t['fondo']};font-family:{o['fuente']};padding:16px;border-radius:{r + 4}px;color:{t['texto']};
                border:1px solid {t['linea']}">
      <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:10px">
        <div><div style="font-size:1.2rem;font-weight:800;color:{t['tinta']}">⚡ Motor de decisión · Bolsa XM</div>
             <div style="font-size:.78rem;color:{t['gris']}">Opción {op} · {o['nombre']} · modo {modo}</div></div>
        <span style="background:{t['acento']};color:{'#0B1F3A' if modo == 'oscuro' and op in 'BF' else '#fff'};
              padding:5px 12px;border-radius:{min(r, 10)}px;font-size:.75rem;font-weight:700">☰ Menú</span></div>
      <div style="display:flex;gap:12px;flex-wrap:wrap">{hero}
        <div style="{caja}padding:10px;flex:1;min-width:250px">
          <div style="font-size:.78rem;font-weight:700;color:{t['tinta']};margin-bottom:2px">Pronóstico de los últimos 3 días</div>
          {grafico_banda(d['serie'], t)}
          <div style="font-size:.66rem;color:{t['gris']}">Línea: mediana · sombra: banda 80 % · punteada: precio real</div></div></div>
      <div style="display:flex;gap:10px;margin:12px 0;flex-wrap:wrap">{tarjetas}</div>
      <div style="{caja}padding:10px 12px">
        <div style="font-size:.78rem;font-weight:700;color:{t['tinta']};margin-bottom:6px">Franja de 24 horas · {d['dia']:%d/%m/%Y}</div>
        <div style="display:flex;gap:3px">{franja}</div>
        <div style="font-size:.78rem;font-weight:700;color:{t['tinta']};margin:12px 0 6px">Mapa de calor día-hora · últimos 14 días</div>
        <div style="display:flex;flex-direction:column;gap:2px">{filas}</div>
        <div style="font-size:.68rem;color:{t['gris']};margin-top:8px">{ley}</div></div>
    </div>"""


def mostrar(op):
    display(HTML(f"""<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(520px,1fr));gap:14px">
                     {vista_previa(op, 'claro')}{vista_previa(op, 'oscuro')}</div>"""))

### Opción A

La paleta vigente, como referencia para comparar. Es sobria y cumple, aunque la combinación verde/rojo es la que peor distinguen las personas con daltonismo (ver el final).

In [4]:
mostrar("A")

### Opción B

Más institucional (sector eléctrico): el ámbar se vuelve acento de marca y por eso **esperar pasa a gris**, lo que además deja que en el mapa resalten solo las horas en que conviene actuar. En oscuro, la mediana va en ámbar sobre azul marino.

In [5]:
mostrar("B")

### Opción C

Minimalista tipo producto SaaS: sin sombras, grises neutros y turquesa. La tarjeta principal se tiñe completa con el color de la señal, lo que la hace muy evidente a distancia.

In [6]:
mostrar("C")

### Opción D

Pensada para accesibilidad. Usa la paleta Okabe-Ito, el estándar en publicaciones científicas para daltonismo: **azul = actuar a favor**, amarillo = esperar, bermellón = evitar. Pierde la convención semáforo verde/rojo, pero la distingue cualquier persona.

In [7]:
mostrar("D")

### Opción E

Tonos apagados (paleta Nord) y bordes finos. Cansa poco la vista en jornadas largas, aunque las señales llaman menos la atención.

In [8]:
mostrar("E")

### Opción F

Estética de instrumentación o SCADA, coherente con el enfoque de señales de la presentación final: esquinas casi rectas, fondo casi negro en oscuro y cifras en monoespaciada, como un panel de medición.

In [9]:
mostrar("F")

## Chequeo 1 · Contraste WCAG

Es la razón de contraste del color sobre el panel. El texto normal necesita **≥ 4.5** (AA) y los elementos gráficos
y el texto grande **≥ 3**. Se mide el texto, el gris secundario y el color de cada señal tal como aparece en el
título de la tarjeta principal.

In [10]:
def luminancia(hexa):
    c = np.array([int(hexa.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)]) / 255
    c = np.where(c <= 0.03928, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    return 0.2126 * c[0] + 0.7152 * c[1] + 0.0722 * c[2]

def contraste(a, b):
    la, lb = sorted([luminancia(a), luminancia(b)], reverse=True)
    return (la + 0.05) / (lb + 0.05)

filas = []
for op, o in OPCIONES.items():
    for modo in ("claro", "oscuro"):
        t = o[modo]
        filas.append({"opción": f"{op} · {o['nombre']}", "modo": modo,
                      **{k: round(contraste(t[k], t["panel"]), 1) for k in ("tinta", "texto", "gris", "verde", "ambar", "rojo")}})
tabla = pd.DataFrame(filas).set_index(["opción", "modo"])
tabla.style.format("{:.1f}").map(lambda v: "background-color:#FECACA" if v < 3 else ("background-color:#FEF3C7" if v < 4.5 else ""))

En la tabla, rojo significa que no alcanza ni para gráficos (< 3) y amarillo que sirve para títulos grandes o gráficos
pero no para texto corto (3 a 4.5). El título de la señal mide 1.75 rem y va en negrita, así que basta con ≥ 3.

## Chequeo 2 · ¿Se distinguen las tres señales con daltonismo?

Simulación de **deuteranopía** y **protanopía**, que afectan a cerca del 6 % de los hombres, con las matrices de
Machado et al. (2009, severidad 1.0). Se aplican a los colores del mapa de calor. A la derecha está la distancia
perceptual **ΔE (CIE76)** entre el par más parecido. Por debajo de ~20, dos señales se confunden a simple vista.

In [11]:
MACHADO = {
    "normal": np.eye(3),
    "deuteranopía": np.array([[0.367322, 0.860646, -0.227968], [0.280085, 0.672501, 0.047413], [-0.011820, 0.042940, 0.968881]]),
    "protanopía": np.array([[0.152286, 1.052583, -0.204868], [0.114503, 0.786281, 0.099216], [-0.003882, -0.048116, 1.051998]]),
}
a_lin = lambda c: np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
a_srgb = lambda c: np.where(c <= 0.0031308, 12.92 * c, 1.055 * np.clip(c, 0, 1) ** (1 / 2.4) - 0.055)

def simular(hexa, m):
    c = np.array([int(hexa.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)]) / 255
    return np.clip(a_srgb(np.clip(m @ a_lin(c), 0, 1)), 0, 1)

def lab(rgb):
    xyz = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]]) @ a_lin(rgb)
    xyz = xyz / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.array([116 * f[1] - 16, 500 * (f[0] - f[1]), 200 * (f[1] - f[2])])

a_hex = lambda rgb: "#" + "".join(f"{int(round(v * 255)):02X}" for v in rgb)
html = ['<table style="border-collapse:collapse;font-size:.82rem">'
        '<tr><th style="text-align:left;padding:4px 10px">Opción · modo</th>'
        + "".join(f'<th style="padding:4px 10px">{v}</th>' for v in MACHADO) + "</tr>"]
for op, o in OPCIONES.items():
    for modo in ("claro", "oscuro"):
        t = o[modo]; celdas = []
        for nombre, m in MACHADO.items():
            cols = [simular(t[f"calor_{k}"], m) for k in ("verde", "ambar", "rojo")]
            de = min(np.linalg.norm(lab(cols[i]) - lab(cols[j])) for i, j in ((0, 1), (0, 2), (1, 2)))
            muestras = "".join(f'<span style="display:inline-block;width:22px;height:16px;background:{a_hex(c)};margin-right:2px"></span>' for c in cols)
            alerta = "color:#B91C1C;font-weight:700" if de < 20 else ""
            celdas.append(f'<td style="padding:4px 10px">{muestras} <span style="{alerta}">ΔE {de:.0f}</span></td>')
        html.append(f'<tr><td style="padding:4px 10px">{op} · {o["nombre"]} · {modo}</td>{"".join(celdas)}</tr>')
display(HTML("".join(html) + "</table>"))

Opción · modo,normal,deuteranopía,protanopía
A · Pizarra (actual) · claro,ΔE 57,ΔE 21,ΔE 27
A · Pizarra (actual) · oscuro,ΔE 69,ΔE 18,ΔE 24
B · Energía · claro,ΔE 57,ΔE 36,ΔE 35
B · Energía · oscuro,ΔE 73,ΔE 48,ΔE 39
C · Grafito · claro,ΔE 79,ΔE 14,ΔE 25
C · Grafito · oscuro,ΔE 69,ΔE 14,ΔE 19
D · Accesible (Okabe-Ito) · claro,ΔE 68,ΔE 34,ΔE 50
D · Accesible (Okabe-Ito) · oscuro,ΔE 59,ΔE 23,ΔE 35
E · Nórdico · claro,ΔE 37,ΔE 7,ΔE 20
E · Nórdico · oscuro,ΔE 27,ΔE 19,ΔE 13


## Cómo elegir

- **Para la presentación final** (enfoque en señales y electrónica), **F** es la que mejor cuenta la historia de "panel
  de instrumentación". **B** es la más institucional.
- **Si importa la accesibilidad**, como criterio de diseño que se puede defender ante el jurado, **D** y **B** son las que mejor pasan el
  chequeo de daltonismo: ΔE mínimo ≥ 23 en todas las variantes. D lo logra cambiando verde/rojo por azul/bermellón; B, porque
  "esperar" en gris se separa por luminosidad. **A** (la actual) queda en el límite, con ΔE 18 en deuteranopía oscuro, y C, E y F
  confunden "esperar" con alguna de las otras en al menos un modo.
- **A** se puede mantener sin cambios. Las demás requieren solo reemplazar los tokens de `TEMAS` en `dashboard/app.py`
  y, para B, C, E y F, cargar la fuente correspondiente de Google Fonts.
- Se pueden mezclar: por ejemplo, la estructura de una opción (tipografía, esquinas, tarjeta) con los colores de otra.